# Phase2: ベースライン（拡張なし）と過学習

**この回で学ぶこと**
- 小さなCNNで画像分類を学習する流れ
- **学習曲線**の読み方と、**過学習**（train は良いのに val が伸びない）の見つけ方
- 少量データでは**シード（乱数）によって結果がぶれる**こと

拡張を試す前に、**拡張なしの基準（ベースライン）**を作ります。これがないと、拡張の効果を測れません。

> ⏱ 所要時間: 約1〜2分（QUICK=True, 60エポック×1シード）

In [ ]:
import sys
from pathlib import Path

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

In [ ]:
QUICK = True                          # True: 1シード（短時間）/ False: 3シード（MANUAL.mdの記載値と同条件）
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = ep(60)

## 1. モデル（SmallCNN）

畳み込み3層（Conv→BatchNorm→ReLU→MaxPool）＋全結合1層の**小さなCNN**です。
意図的に小さくして、CPU/MPSで数十秒で学習でき、過学習の様子が見やすいようにしています。

In [ ]:
model = T.SmallCNN(2)
print(model)
print("パラメータ数:", sum(p.numel() for p in model.parameters()))

## 2. 学習する（拡張なし）

`aug="none"` は**リサイズと正規化のみ**です。`fit()` は学習を1回行い、エポックごとの履歴を返します。
`test` 精度は**最終エポックのモデルで1回だけ**評価します（testの結果でエポックを選ぶと、testが学習に混ざってしまうため）。

In [ ]:
r = fit(aug="none", seed=0, epochs=EPOCHS, verbose=False)
plot_history(r)
summarize(r)

## 3. 学習曲線の読み方

| 見るもの | 過学習のサイン |
|---|---|
| **loss** | train loss は下がり続けるのに、**val loss が下がらない/上がる** |
| **精度** | train 精度が val 精度を**大きく上回る**（差 = gap） |

val が約76枚と少ないため、val の線は**ギザギザ**します。1枚の判定が変わるだけで約1.3ポイント動きます。

## 4. シードを変えるとどうなる？

乱数シード（初期値・データの順序）を変えるだけで結果がぶれます。**条件の比較は複数シードの平均で行う**理由です。

In [ ]:
df, results = run_seeds(SEEDS, aug="none", epochs=EPOCHS)
df

In [ ]:
mean_table(df)

## 🧪 試してみよう
1. `EPOCHS` を 20 / 60 / 150 に変えて学習曲線を見比べよう。エポックを増やすと gap はどう変わる？
2. `QUICK = False` で3シードにして、標準偏差（±）を確認しよう。val と test のどちらがぶれやすい？
3. `fit()` に `lr=1e-2` や `lr=1e-4` を渡して、学習の進み方の違いを見よう。
4. `src/train.py` の `SmallCNN` のチャンネル数（16/32/64）を大きくすると、過学習はどうなる？

## まとめ（MANUAL.mdの記載値）
- 拡張なし（60エポック、3シード平均）: train 0.859 / val 0.719 / test 0.809、gap 約0.14
- **train loss が下がるのに val loss が下がらない = 過学習**。ここを拡張で改善するのが以降のテーマ。
- 少量データでは結果がぶれる。**複数シード平均で比べる。**

➡ 次は `03_augment_vs_none.ipynb`（拡張あり vs なしの比較）